In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
# import seaborn as sns
import shap
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
y_col = "Churn Label"
list_use_cols = [
    "Tenure Months", "Monthly Charges", "City", "Gender", "Senior Citizen", "Partner",
    "Dependents", "Phone Service", "Multiple Lines", "Internet Service", "Online Security",
    "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method","Total Charges", "Churn Label"
]

## 1. データ読み込み

In [ ]:
# 01_eda.ipynbで一部前処理をして保存したデータの読み込み

df_data = pd.read_csv("./data/prepro_data.csv", encoding="utf-8")

In [ ]:
df_data.head(3)

In [ ]:
df_data.dtypes

## 2. 前処理

In [ ]:
df_prepro = df_data[list_use_cols].copy()

In [ ]:
list_non_numeric_cols = df_prepro.select_dtypes(exclude="number").columns.tolist()


In [ ]:
# カテゴリ型に変更
for col in list_non_numeric_cols:
    df_prepro[col] = df_prepro[col].astype("category")

In [ ]:
# 数値データの離散化
# 木モデルなので厳密には不要だが、
# ノイズを減らし解釈しやすくするため離散化

# Total Chargesは一旦100区切り。
# lightgbmなのでスケーリング必須ではないが、場合によっては対数変換と標準化を入れる。
df_prepro["Total Charges"] = (df_prepro["Total Charges"]//100) * 100

# Tenure Months は一旦５刻み
df_prepro["Tenure Months"] = (df_prepro["Tenure Months"]//5) * 5

# Monthly Chargesは一旦5刻み
df_prepro["Monthly Charges"] = (df_prepro["Monthly Charges"]//5) * 5


In [ ]:
# edaより、付随サービスに入っているかどうかがチャーンに影響がありそうだったため、フラグを作成
# # Online Security / Online Backup /
# Device Protection / Tech Support の
# いずれかを契約している場合True

df_prepro["is_optional_service"] = ~(
    (df_prepro["Online Security"] == "No") & 
    (df_prepro["Online Backup"] == "No") & 
    (df_prepro["Device Protection"] == "No") & 
    (df_prepro["Tech Support"] == "No")
)

In [ ]:
df_prepro.head(3)

## 3. lighgbmの学習とfeature_importanceの確認

In [ ]:
# 学習前にデータ型の再確認
df_prepro.dtypes

### 3.1 データセットの作成

In [ ]:
list_features = list_use_cols + ["is_optional_service"]
list_features.remove(y_col)

In [ ]:
X = df_prepro[list_features]
y = df_prepro[y_col]

In [ ]:
# テストデータの20%を切り出す

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=seed, 
    stratify=y
)

In [ ]:
# 検証データを全体の20％にするため、80%のうちの20%を指定

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, 
    test_size=0.2, 
    random_state=seed, 
    stratify=y_train_val
)


### 3.2　lightgbmの学習と精度の確認

In [ ]:
model_lgb = lgb.LGBMClassifier(
    verbose=-1,
    random_state=seed,
    importance_type="gain"
)

In [ ]:
model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), 
        
    ]
)

In [ ]:
y_pred = model_lgb.predict(X_test)

In [ ]:
# accuracy_score, precision_score, recall_score

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="Yes")
recall= recall_score(y_test, y_pred, pos_label="Yes")

print(f"Metrics scores:\nacc:{acc}\nprecision:{precision}\nrecall:{recall}")

In [ ]:
# 予測精度が劣化しすぎていると、feature_importanceとshapを見ても妥当性に疑問が残るので、精度も確認。
# 精度としては下記で、良くも悪くもなさそうだけど、feature_importanceとshapを見る分には致命的な精度ではなさそう？

# Scores:
# acc:0.7967306325515281
# precision:0.6341463414634146
# recall:0.5561497326203209

## 4. 特徴量の影響度の確認

### 4.1 feature_importance

In [ ]:
importances = model_lgb.feature_importances_

In [ ]:
feature_imp = pd.DataFrame({
    "Feature": model_lgb.booster_.feature_name(),
    "Importance": importances
})

# 重要度順にソートしてプロット
feature_imp.sort_values(by="Importance", ascending=False, inplace=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_imp["Feature"].head(10), feature_imp["Importance"][:10])
plt.xlabel("Gain Feature Importance")
plt.ylabel("Features")
plt.gca().invert_yaxis()
plt.show()

# Contract が最も重要な特徴量となった。
# 長期契約ほど解約率が低くなることを反映している可能性がある。

### 4.2 shap値

In [ ]:
explainer = shap.TreeExplainer(model_lgb)

shap_values = explainer.shap_values(X_test)

if hasattr(shap_values, "values"):
    # 新しいSHAPオブジェクト（Explanationオブジェクト）の場合
    shap_matrix = shap_values.values
else:
    # 従来のNumPy配列の場合
    shap_matrix = shap_values

# df_shap = pd.DataFrame(shap_matrix, columns=X_test.columns)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_matrix, X_test, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# dependence_plot
# 影響が大きかったContractを確認

shap.dependence_plot(
    ind="Contract",
    shap_values=shap_values, 
    features=X_test,
)